# Deep dive 8/13: `call_center_interactions`

Second fact table. Dictionary declares 800,000 rows, Source: Contact Center, Partition: daily.

`table_specs.py` has a specific open question baked into its comment above this spec: **`contact_reason` and `reason_category` showed byte-for-byte identical value distributions in the original Bronze profile** (same 6 categories, same row counts down to the last digit) -- strong evidence they're the same value stored twice, but the profile only confirmed matching *aggregate* distributions, not a row-by-row identity check. The comment literally asks for `SELECT count(*) FROM silver.fact_call_center_interactions WHERE contact_reason <> reason_category` before trusting either is safe to drop -- that's section 7 here.

Dictionary NOT NULL: `interaction_id`, `interaction_date`, `process_date`, `customer_id`, `interaction_type`, `channel`, `contact_reason`, `reason_category`, `requires_followup`, `was_escalated`, `has_transcript`, `has_recording`. `contracts.py`'s current `required` tuple has 11 of these 12 -- **`has_recording` is missing**, checked as a fix candidate in section 3. Nullable: `agent_id` (a nullable FK -- not every interaction has an assigned agent), `duration_seconds`, `wait_time_seconds`, `was_resolved`, `detected_sentiment`, `sentiment_score`, `customer_detected_accent`, `agent_used_accent`, `mentioned_products`.

`channel` is already gated in `contracts.py` with 6 values (`Phone, Web Chat, WhatsApp, Email, App, Web`) -- one more than the dictionary's declared 5 (`Phone, Web Chat, WhatsApp, Email, App`). Section 5 confirms whether the extra `Web` value is real and whether the existing gate is still accurate.

Run top to bottom. Checklist near the end; free cells after that.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. Sample rows -- Bronze and Silver side by side

In [2]:
sample_ids = q("SELECT interaction_id FROM bronze.call_center_interactions ORDER BY interaction_id LIMIT 20")["interaction_id"].tolist()
ids_sql = ", ".join(f"'{i}'" for i in sample_ids)

print("Bronze:")
display(q_full(f"SELECT * FROM bronze.call_center_interactions WHERE interaction_id IN ({ids_sql}) ORDER BY interaction_id"))

Bronze:


,interaction_id,interaction_date,process_date,customer_id,agent_id,interaction_type,channel,contact_reason,reason_category,duration_seconds,wait_time_seconds,was_resolved,requires_followup,detected_sentiment,sentiment_score,customer_detected_accent,agent_used_accent,was_escalated,mentioned_products,has_transcript,has_recording,_source_file,_ingested_at,_source_table,day,month,year
0,INT-0003AGRO12D5GXCQ,2026-05-27 01:03:33,2026-05-26,CLI-VSXUZM56BILD,AGT-JNY9R22CQM,Inbound Call,Phone,Transaccional,Transaccional,105.0,44.0,True,False,Neutral,-0.3,argentine,argentine,False,NaN,True,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/call_center_interactions/year=2026...,2026-09-28 19:30:36.021752-03:00,call_center_interactions,26,05,2026
1,INT-0007KT4KOKT4PFCN,2024-01-15 10:03:01,2024-01-15,CLI-MV5UYGFCLHSZ,AGT-3XZPJ3T8CP,Inbound Call,Phone,Comercial,Comercial,488.0,111.0,True,False,Positivo,0.57,NaN,NaN,False,NaN,True,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/call_center_interactions/year=2024...,2026-09-28 19:30:36.021752-03:00,call_center_interactions,15,01,2024
2,INT-00082S3TI0HXCT50,2026-05-18 22:59:02,2026-05-18,CLI-BRIPZEZSVD7D,AGT-YEHDH59WDI,Outbound Call,Phone,Retención,Retención,532.0,NaN,True,True,Muy Positivo,0.78,mexican,mexican,True,"PRD-5JM20QHJ50ZQ,PRD-CLX0M78LQCTI,PRD-IJ7XUZRJCZ6O",False,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/call_center_interactions/year=2026...,2026-09-28 19:30:36.021752-03:00,call_center_interactions,18,05,2026
3,INT-0008WBTYMYEMLMBS,2026-02-20 09:15:18,2026-02-20,CLI-P2664O3F9CW5,AGT-E76DUY8F1G,Outbound Call,Phone,Transaccional,Transaccional,201.0,NaN,True,True,Neutral,-0.09,mexican,mexican,False,"PRD-2QIXXAJ3ZP62,PRD-AZJYYFBQDMGR",False,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/call_center_interactions/year=2026...,2026-09-28 19:30:36.021752-03:00,call_center_interactions,20,02,2026
4,INT-000AU452XMN2Z7SR,2025-09-29 22:52:59,2025-09-29,CLI-T8T6LY40X9QO,AGT-TSMQUDNLVQ,Inbound Call,Phone,Técnico,Técnico,286.0,68.0,True,False,Positivo,0.31,NaN,NaN,False,NaN,True,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/call_center_interactions/year=2025...,2026-09-28 19:30:36.021752-03:00,call_center_interactions,29,09,2025
5,INT-000D0P01K7IBJS61,2025-08-04 09:33:35,2025-08-04,CLI-PFR13H57XW73,AGT-NUS3RT0T0E,Inbound Call,Phone,Técnico,Técnico,155.0,86.0,True,False,Neutral,0.24,NaN,NaN,False,NaN,True,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/call_center_interactions/year=2025...,2026-09-28 19:30:36.021752-03:00,call_center_interactions,04,08,2025
6,INT-000LP5LMI66APS7R,2024-09-06 06:08:09,2024-09-05,CLI-7L2CPVKU44PZ,AGT-TSSIHYMD02,Inbound Call,Phone,Transaccional,Transaccional,286.0,67.0,True,False,Neutral,-0.04,NaN,NaN,False,NaN,False,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/call_center_interactions/year=2024...,2026-09-28 19:30:36.021752-03:00,call_center_interactions,05,09,2024
7,INT-000MUYBH4CTIWLVP,2024-02-16 20:31:04,2024-02-16,CLI-OM1LVXUQS3Z7,AGT-7STCFNILU3,Inbound Call,Phone,Producto,Producto,321.0,68.0,True,False,Neutral,0.11,mexican,mexican,False,NaN,False,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/call_center_interactions/year=2024...,2026-09-28 19:30:36.021752-03:00,call_center_interactions,16,02,2024
8,INT-000O756KFLVCX95U,2024-08-14 06:48:26,2024-08-13,CLI-K7GQ2Z7FSNUW,AGT-EI5HV76G6F,Inbound Call,Phone,Transaccional,Transaccional,108.0,83.0,True,False,Neutral,0.08,NaN,NaN,True,NaN,False,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/data/call_center_interactions/year=2024...,2026-09-28 19:30:36.021752-03:00,call_center_interactions,13,08,2024
9,INT-000V6BOKJE5STGSG,2024-09-20 00:02:32,2024-09-19,CLI-D9KQXSO62V6A,AGT-K5E9XE6DZU,Outbound Call,Phone,Queja,Queja,109.0,NaN,True,False,Muy Negativo,-0.92,mexican,mexican,False,NaN,False,True,s3://factored-datathon-2026-s3-157725502942-us-east-2-an/d

In [3]:
print("Silver:")
display(q_full(f"SELECT * FROM silver.fact_call_center_interactions WHERE interaction_id IN ({ids_sql}) ORDER BY interaction_id"))

Silver:


,interaction_id,interaction_date,process_date,customer_id,agent_id,interaction_type,channel,contact_reason,reason_category,duration_seconds,wait_time_seconds,was_resolved,requires_followup,detected_sentiment,sentiment_score,customer_detected_accent,agent_used_accent,was_escalated,mentioned_products,has_transcript,has_recording
0,INT-0003AGRO12D5GXCQ,2026-05-27 01:03:33,2026-05-26,CLI-VSXUZM56BILD,AGT-JNY9R22CQM,Inbound Call,Phone,Transaccional,Transaccional,105.0,44.0,True,False,Neutral,-0.30,argentine,argentine,False,NaN,True,True
1,INT-0007KT4KOKT4PFCN,2024-01-15 10:03:01,2024-01-15,CLI-MV5UYGFCLHSZ,AGT-3XZPJ3T8CP,Inbound Call,Phone,Comercial,Comercial,488.0,111.0,True,False,Positivo,0.57,NaN,NaN,False,NaN,True,True
2,INT-00082S3TI0HXCT50,2026-05-18 22:59:02,2026-05-18,CLI-BRIPZEZSVD7D,AGT-YEHDH59WDI,Outbound Call,Phone,Retención,Retención,532.0,NaN,True,True,Muy Positivo,0.78,mexican,mexican,True,"PRD-5JM20QHJ50ZQ,PRD-CLX0M78LQCTI,PRD-IJ7XUZRJCZ6O",False,True
3,INT-0008WBTYMYEMLMBS,2026-02-20 09:15:18,2026-02-20,CLI-P2664O3F9CW5,AGT-E76DUY8F1G,Outbound Call,Phone,Transaccional,Transaccional,201.0,NaN,True,True,Neutral,-0.09,mexican,mexican,False,"PRD-2QIXXAJ3ZP62,PRD-AZJYYFBQDMGR",False,True
4,INT-000AU452XMN2Z7SR,2025-09-29 22:52:59,2025-09-29,CLI-T8T6LY40X9QO,AGT-TSMQUDNLVQ,Inbound Call,Phone,Técnico,Técnico,286.0,68.0,True,False,Positivo,0.31,NaN,NaN,False,NaN,True,True
5,INT-000D0P01K7IBJS61,2025-08-04 09:33:35,2025-08-04,CLI-PFR13H57XW73,AGT-NUS3RT0T0E,Inbound Call,Phone,Técnico,Técnico,155.0,86.0,True,False,Neutral,0.24,NaN,NaN,False,NaN,True,True
6,INT-000LP5LMI66APS7R,2024-09-06 06:08:09,2024-09-05,CLI-7L2CPVKU44PZ,AGT-TSSIHYMD02,Inbound Call,Phone,Transaccional,Transaccional,286.0,67.0,True,False,Neutral,-0.04,NaN,NaN,False,NaN,False,True
7,INT-000MUYBH4CTIWLVP,2024-02-16 20:31:04,2024-02-16,CLI-OM1LVXUQS3Z7,AGT-7STCFNILU3,Inbound Call,Phone,Producto,Producto,321.0,68.0,True,False,Neutral,0.11,mexican,mexican,False,NaN,False,True
8,INT-000O756KFLVCX95U,2024-08-14 06:48:26,2024-08-13,CLI-K7GQ2Z7FSNUW,AGT-EI5HV76G6F,Inbound Call,Phone,Transaccional,Transaccional,108.0,83.0,True,False,Neutral,0.08,NaN,NaN,True,NaN,False,True
9,INT-000V6BOKJE5STGSG,2024-09-20 00:02:32,2024-09-19,CLI-D9KQXSO62V6A,AGT-K5E9XE6DZU,Outbound Call,Phone,Queja,Queja,109.0,NaN,True,False,Muy Negativo,-0.92,mexican,mexican,False,NaN,False,True


## 1. Column inventory -- names AND types, Bronze vs. Silver

In [4]:
expected_silver_columns = [
    "interaction_id", "interaction_date", "process_date", "customer_id", "agent_id",
    "interaction_type", "channel", "contact_reason", "reason_category", "duration_seconds",
    "wait_time_seconds", "was_resolved", "requires_followup", "detected_sentiment",
    "sentiment_score", "customer_detected_accent", "agent_used_accent", "was_escalated",
    "mentioned_products", "has_transcript", "has_recording",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_call_center_interactions'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.fact_call_center_interactions:", missing or "none")
print("Extra in silver.fact_call_center_interactions:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

Missing from silver.fact_call_center_interactions: none
Extra in silver.fact_call_center_interactions: none

Column order as built: ['interaction_id', 'interaction_date', 'process_date', 'customer_id', 'agent_id', 'interaction_type', 'channel', 'contact_reason', 'reason_category', 'duration_seconds', 'wait_time_seconds', 'was_resolved', 'requires_followup', 'detected_sentiment', 'sentiment_score', 'customer_detected_accent', 'agent_used_accent', 'was_escalated', 'mentioned_products', 'has_transcript', 'has_recording']


In [5]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='call_center_interactions' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_call_center_interactions' ORDER BY ordinal_position
"""))

Bronze columns + types:


,column_name,data_type
0,interaction_id,VARCHAR
1,interaction_date,VARCHAR
2,process_date,VARCHAR
3,customer_id,VARCHAR
4,agent_id,VARCHAR
5,interaction_type,VARCHAR
6,channel,VARCHAR
7,contact_reason,VARCHAR
8,reason_category,VARCHAR
9,duration_seconds,VARCHAR


Silver columns + types:


,column_name,data_type
0,interaction_id,VARCHAR
1,interaction_date,TIMESTAMP
2,process_date,DATE
3,customer_id,VARCHAR
4,agent_id,VARCHAR
5,interaction_type,VARCHAR
6,channel,VARCHAR
7,contact_reason,VARCHAR
8,reason_category,VARCHAR
9,duration_seconds,DOUBLE


## 2. Row-count reconciliation + dedup

In [6]:
bronze_total = q("SELECT count(*) AS n FROM bronze.call_center_interactions")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT interaction_id) AS n FROM bronze.call_center_interactions")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.fact_call_center_interactions")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT interaction_id) AS n FROM silver.fact_call_center_interactions")["n"][0]

print(f"bronze.call_center_interactions:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct interaction_id")
print(f"silver.fact_call_center_interactions: {silver_total:,} rows, {silver_distinct_pk:,} distinct interaction_id")
print()
print("Dictionary declares 800,000 rows -- matches" if bronze_total == 800_000 else f"Dictionary declares 800,000 rows -- MISMATCH, Bronze has {bronze_total:,}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

bronze.call_center_interactions:      686,296 rows, 686,296 distinct interaction_id
silver.fact_call_center_interactions: 686,296 rows, 686,296 distinct interaction_id

Dictionary declares 800,000 rows -- MISMATCH, Bronze has 686,296
Duplicate raw rows in Bronze: 0
OK -- Silver row count matches Bronze distinct PK count


## 3. NOT NULL audit -- Bronze raw vs. Silver typed, all 12 dictionary-declared columns

Including `has_recording`, missing from `contracts.py`'s `required` tuple -- checked here as a fix candidate.

In [7]:
declared_not_null = [
    "interaction_id", "interaction_date", "process_date", "customer_id", "interaction_type",
    "channel", "contact_reason", "reason_category", "requires_followup", "was_escalated",
    "has_transcript", "has_recording",
]
currently_required = {"interaction_id","interaction_date","process_date","customer_id","interaction_type",
                       "channel","contact_reason","reason_category","requires_followup","was_escalated","has_transcript"}

rows = []
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.call_center_interactions
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.fact_call_center_interactions WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "in_contracts_required": col in currently_required,
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

,column,declared,in_contracts_required,bronze_nulls,silver_nulls,cast_introduced_nulls
0,interaction_id,NOT NULL,True,0,0,0
1,interaction_date,NOT NULL,True,0,0,0
2,process_date,NOT NULL,True,0,0,0
3,customer_id,NOT NULL,True,0,0,0
4,interaction_type,NOT NULL,True,0,0,0
5,channel,NOT NULL,True,0,0,0
6,contact_reason,NOT NULL,True,0,0,0
7,reason_category,NOT NULL,True,0,0,0
8,requires_followup,NOT NULL,True,0,0,0
9,was_escalated,NOT NULL,True,0,0,0


## 4. Nullable columns -- population rate, Bronze vs. Silver

In [8]:
nullable_cols = ["agent_id", "duration_seconds", "wait_time_seconds", "was_resolved",
                  "detected_sentiment", "sentiment_score", "customer_detected_accent",
                  "agent_used_accent", "mentioned_products"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.call_center_interactions")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.call_center_interactions
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.fact_call_center_interactions WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total,
                 "bronze_populated": bronze_populated, "bronze_pct": round(100*bronze_populated/total, 2),
                 "silver_populated": silver_populated, "silver_pct": round(100*silver_populated/total, 2)})

pd.DataFrame(rows)

,column,total,bronze_populated,bronze_pct,silver_populated,silver_pct
0,agent_id,686296,686296,100.00,686296,100.00
1,duration_seconds,686296,590062,85.98,590062,85.98
2,wait_time_seconds,686296,480678,70.04,480678,70.04
3,was_resolved,686296,686296,100.00,686296,100.00
4,detected_sentiment,686296,686296,100.00,686296,100.00
5,sentiment_score,686296,686296,100.00,686296,100.00
6,customer_detected_accent,686296,481546,70.17,481546,70.17
7,agent_used_accent,686296,481546,70.17,481546,70.17
8,mentioned_products,686296,274341,39.97,274341,39.97


## 5. Domain checks -- `interaction_type`, `reason_category`, `detected_sentiment`, and re-verifying the already-gated `channel`

In [9]:
# interaction_type -- dictionary: Inbound Call, Outbound Call, Chat, Email, Video
print("Bronze:")
display(q("SELECT interaction_type, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))
print("Silver:")
display(q("SELECT interaction_type, count(*) AS n FROM silver.fact_call_center_interactions GROUP BY 1 ORDER BY n DESC"))

Bronze:


,interaction_type,n
0,Inbound Call,480678
1,Outbound Call,102572
2,Chat,68691
3,Email,27543
4,Video,6812


Silver:


,interaction_type,n
0,Inbound Call,480678
1,Outbound Call,102572
2,Chat,68691
3,Email,27543
4,Video,6812


In [10]:
# reason_category -- dictionary text was truncated in the PDF table, so discovering the real
# domain directly rather than assuming.
print("Bronze:")
display(q("SELECT reason_category, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))

Bronze:


,reason_category,n
0,Transaccional,240056
1,Producto,150863
2,Queja,117021
3,Técnico,102899
4,Comercial,54879
5,Retención,20578


In [11]:
# detected_sentiment -- dictionary: Positive, Neutral, Negative, Very Negative
print("Bronze:")
display(q("SELECT detected_sentiment, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))

Bronze:


,detected_sentiment,n
0,Neutral,459712
1,Negativo,94322
2,Positivo,75562
3,Muy Negativo,37727
4,Muy Positivo,18973


In [12]:
# channel -- already gated in contracts.py as {Phone, Web Chat, WhatsApp, Email, App, Web}
# (6 values -- one more than the dictionary's declared 5). Confirming the gate is still accurate.
gated_channel_domain = {"Phone", "Web Chat", "WhatsApp", "Email", "App", "Web"}
real_channel = set(q("SELECT DISTINCT channel AS v FROM bronze.call_center_interactions")["v"])
print("channel values NOT covered by the current contracts.py domain:", real_channel - gated_channel_domain or "none -- gate is still accurate")
display(q("SELECT channel, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))

channel values NOT covered by the current contracts.py domain: none -- gate is still accurate


,channel,n
0,Phone,583250
1,Email,27543
2,App,26364
3,WhatsApp,22888
4,Web Chat,22856
5,Web,3395


## 6. FK integrity -- `customer_id` (required) and `agent_id` (nullable), reproduced directly

In [13]:
fk_checks = [
    ("customer_id", "silver.dim_customers", "customer_id"),
    ("agent_id", "silver.dim_service_agents", "agent_id"),
]
rows = []
for child_col, parent_table, parent_col in fk_checks:
    total = q(f"SELECT count(*) AS n FROM silver.fact_call_center_interactions WHERE {child_col} IS NOT NULL")["n"][0]
    orphans = q(f"""
        SELECT count(*) AS n FROM silver.fact_call_center_interactions t
        WHERE t.{child_col} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table} p WHERE p.{parent_col} = t.{child_col})
    """)["n"][0]
    rows.append({"fk": f"{child_col} -> {parent_table}.{parent_col}", "populated": total,
                 "orphans": orphans, "orphan_pct": round(100*orphans/total, 4) if total else None})

pd.DataFrame(rows)

,fk,populated,orphans,orphan_pct
0,customer_id -> silver.dim_customers.customer_id,686296,0,0.0
1,agent_id -> silver.dim_service_agents.agent_id,686296,0,0.0


## 7. `contact_reason` vs. `reason_category` -- the exact identity check `table_specs.py`'s comment asks for

The comment above the spec flags matching aggregate distributions between these two columns as suspicious but unconfirmed at the row level. Running the exact query it names.

In [14]:
mismatch_count = q("SELECT count(*) AS n FROM silver.fact_call_center_interactions WHERE contact_reason <> reason_category")["n"][0]
total = q("SELECT count(*) AS n FROM silver.fact_call_center_interactions")["n"][0]
print(f"Rows where contact_reason <> reason_category: {mismatch_count:,} / {total:,}")
print("These ARE the same value stored twice" if mismatch_count == 0 else "These are NOT identical -- some real divergence exists")

Rows where contact_reason <> reason_category: 0 / 686,296
These ARE the same value stored twice


In [15]:
# If there's any divergence, show a sample of the mismatching rows to see the pattern.
q_full("""
    SELECT interaction_id, contact_reason, reason_category
    FROM silver.fact_call_center_interactions
    WHERE contact_reason <> reason_category
    LIMIT 20
""")

,interaction_id,contact_reason,reason_category


## 8. `duration_seconds`/`wait_time_seconds` -- range sanity

In [16]:
q("""
    SELECT
        typeof(duration_seconds) AS duration_type, typeof(wait_time_seconds) AS wait_type,
        min(duration_seconds) AS min_duration, max(duration_seconds) AS max_duration,
        min(wait_time_seconds) AS min_wait, max(wait_time_seconds) AS max_wait,
        count(*) FILTER (WHERE duration_seconds < 0) AS negative_duration,
        count(*) FILTER (WHERE wait_time_seconds < 0) AS negative_wait
    FROM silver.fact_call_center_interactions
    GROUP BY ALL
""")

,duration_type,wait_type,min_duration,max_duration,min_wait,max_wait,negative_duration,negative_wait
0,DOUBLE,DOUBLE,30.0,1204.0,0.0,424.0,0,0


## 9. `sentiment_score` -- range sanity (dictionary: -1 to 1) + cross-check against `detected_sentiment`

In [17]:
q("""
    SELECT
        typeof(sentiment_score) AS silver_type,
        min(sentiment_score) AS min_score, max(sentiment_score) AS max_score,
        count(*) FILTER (WHERE sentiment_score < -1 OR sentiment_score > 1) AS out_of_range,
        count(*) FILTER (WHERE sentiment_score IS NOT NULL) AS populated, count(*) AS total
    FROM silver.fact_call_center_interactions
    GROUP BY ALL
""")

,silver_type,min_score,max_score,out_of_range,populated,total
0,DOUBLE,-1.0,1.0,0,686296,686296


In [18]:
# Does a higher sentiment_score line up with a more positive detected_sentiment?
q("""
    SELECT detected_sentiment, count(*) AS n,
           round(avg(sentiment_score), 3) AS avg_score,
           round(min(sentiment_score), 3) AS min_score, round(max(sentiment_score), 3) AS max_score
    FROM silver.fact_call_center_interactions
    WHERE sentiment_score IS NOT NULL
    GROUP BY 1
    ORDER BY avg_score DESC
""")

,detected_sentiment,n,avg_score,min_score,max_score
0,Muy Positivo,18973,0.850,0.7,1.0
1,Positivo,75562,0.500,0.3,0.7
2,Neutral,459712,0.000,-0.3,0.3
3,Negativo,94322,-0.500,-0.7,-0.3
4,Muy Negativo,37727,-0.849,-1.0,-0.7


## 10. Boolean columns -- cast correctness + cross-field logic

`was_resolved`, `requires_followup`, `was_escalated`, `has_transcript`, `has_recording`.

In [19]:
q("""
    SELECT
        typeof(was_resolved) AS t1, typeof(requires_followup) AS t2, typeof(was_escalated) AS t3,
        typeof(has_transcript) AS t4, typeof(has_recording) AS t5,
        count(*) FILTER (WHERE was_resolved) AS resolved_true,
        count(*) FILTER (WHERE was_resolved IS NULL) AS resolved_null,
        count(*) FILTER (WHERE requires_followup) AS followup_true,
        count(*) FILTER (WHERE was_escalated) AS escalated_true,
        count(*) FILTER (WHERE has_transcript) AS transcript_true,
        count(*) FILTER (WHERE has_recording) AS recording_true,
        count(*) AS total
    FROM silver.fact_call_center_interactions
    GROUP BY ALL
""")

,t1,t2,t3,t4,t5,resolved_true,resolved_null,followup_true,escalated_true,transcript_true,recording_true,total
0,BOOLEAN,BOOLEAN,BOOLEAN,BOOLEAN,BOOLEAN,526030,0,239054,68386,171321,590062,686296


In [20]:
# Does was_resolved = False correlate with requires_followup = True, as you'd expect operationally?
q("""
    SELECT was_resolved, requires_followup, count(*) AS n
    FROM silver.fact_call_center_interactions
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

,was_resolved,requires_followup,n
0,False,True,160266
1,True,False,447242
2,True,True,78788


## 11. `customer_detected_accent`/`agent_used_accent` -- domain spot check

In [21]:
print("customer_detected_accent:")
display(q("SELECT customer_detected_accent, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))
print("agent_used_accent:")
display(q("SELECT agent_used_accent, count(*) AS n FROM bronze.call_center_interactions GROUP BY 1 ORDER BY n DESC"))

customer_detected_accent:


,customer_detected_accent,n
0,mexican,240674
1,NaN,204750
2,colombian,144712
3,argentine,96160


agent_used_accent:


,agent_used_accent,n
0,mexican,241931
1,NaN,204750
2,colombian,144386
3,argentine,95229


## 12. `mentioned_products` -- population rate + format spot check

In [22]:
total = q("SELECT count(*) AS n FROM bronze.call_center_interactions")["n"][0]
populated = q("SELECT count(*) AS n FROM bronze.call_center_interactions WHERE mentioned_products IS NOT NULL")["n"][0]
print(f"mentioned_products populated: {populated:,} / {total:,} = {100*populated/total:.2f}%")
print()
print("Sample values:")
display(q_full("SELECT mentioned_products FROM bronze.call_center_interactions WHERE mentioned_products IS NOT NULL LIMIT 10"))

mentioned_products populated: 274,341 / 686,296 = 39.97%

Sample values:


,mentioned_products
0,"PRD-LWE7ZI313MSF,PRD-2DZJYU0POJ9N,PRD-4FGYV9I6WVAN"
1,"PRD-0BOYEP2OM3RU,PRD-L4ZSCFDICUT3,PRD-YMTPFQY9TJHF"
2,"PRD-EHNER1394RVT,PRD-0VG1IMHF194A"
3,PRD-MMOPNUMWNBFK
4,PRD-RA4HF4J8KOCE
5,"PRD-DD67YH842U3R,PRD-UV1CR4D5TRT5,PRD-FEO8FAQGJBTG"
6,PRD-2NBLFOC1VCZ9
7,"PRD-JNA8TEGO5EGC,PRD-5VBZCERI384G,PRD-51YVFYWTXP65"
8,"PRD-TML43DZKD5LE,PRD-11PSI5W5XBEM"
9,"PRD-83BHO4YZ5NVH,PRD-NRAA2RN8QU1X,PRD-FUX1US6QVA1A"


## 13. `interaction_date`/`process_date` -- consistency + window check

Same checks run on `transactions`' `transaction_date`/`process_date` pair, since that table showed a real ~25% one-day rollback pattern worth re-checking here.

In [23]:
q("""
    SELECT
        typeof(interaction_date) AS interaction_date_type, typeof(process_date) AS process_date_type,
        min(interaction_date) AS earliest_interaction, max(interaction_date) AS latest_interaction,
        min(process_date) AS earliest_process, max(process_date) AS latest_process,
        count(*) FILTER (WHERE process_date <> CAST(interaction_date AS DATE)) AS process_date_differs,
        count(*) FILTER (WHERE process_date < CAST(interaction_date AS DATE)) AS processed_before,
        count(*) FILTER (WHERE process_date > CAST(interaction_date AS DATE)) AS processed_after
    FROM silver.fact_call_center_interactions
    GROUP BY ALL
""")

,interaction_date_type,process_date_type,earliest_interaction,latest_interaction,earliest_process,latest_process,process_date_differs,processed_before,processed_after
0,TIMESTAMP,DATE,2023-06-17 08:03:26,2026-06-18 07:58:13,2023-06-17,2026-06-17,228318,228318,0


In [24]:
# Same window check as every other table.
q("""
    SELECT
        count(*) FILTER (WHERE interaction_date < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE interaction_date > TIMESTAMP '2026-06-17 23:59:59') AS after_window
    FROM silver.fact_call_center_interactions
""")

,before_window,after_window
0,0,175


## Checklist

- [ ] Column inventory (1): all 21 Silver columns present, types correct
- [ ] Row reconciliation (2): 800,000 rows as the dictionary declares, dedup clean
- [ ] NOT NULL audit (3): all 12 declared columns populated -- note whether `has_recording` is clean (fix candidate)
- [ ] Nullable columns (4): population rates make sense
- [ ] Domain checks (5): `interaction_type`/`reason_category`/`detected_sentiment` real values, `channel` gate re-verified
- [ ] FK integrity (6): `customer_id`/`agent_id` orphan rates reproduced directly
- [ ] `contact_reason` vs. `reason_category` (7): resolves the open question from `table_specs.py`'s comment -- are they really identical row-for-row?
- [ ] `duration_seconds`/`wait_time_seconds` (8): no negatives
- [ ] `sentiment_score` (9): within -1 to 1, lines up sensibly with `detected_sentiment`
- [ ] Boolean columns (10): cast correctness, `was_resolved`/`requires_followup` shape makes sense
- [ ] Accent columns (11): domain sanity
- [ ] `mentioned_products` (12): population/format sanity
- [ ] Dates (13): `process_date` vs. `interaction_date` consistency, window check

In [25]:
con.close()
print("closed")

closed
